# Med-Drishti: Medical OCR & Prescription Fine-Tuning on Kaggle GPU

This notebook fine-tunes **TrOCR (Transformer OCR)** (`microsoft/trocr-base-stage1` or `microsoft/trocr-base-printed`) for handwritten prescription recognition and medical document entity extraction.

### Kaggle Setup Instructions:
1. Accelerator: Select **GPU T4 x2** or **GPU P100** under Notebook Settings.
2. Internet: Turn **ON** to download HuggingFace transformers and datasets.
3. Persistence: Turn **Filesystem ON** to retain output models.

In [ ]:
# Step 1: Install necessary libraries
import sys
import subprocess

packages = ['transformers', 'datasets', 'evaluate', 'jiwer', 'accelerate', 'albumentations', 'torch', 'torchvision', 'pillow', 'pandas', 'opencv-python']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    TrOCRProcessor, 
    VisionEncoderDecoderModel, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    default_data_collator
)
import evaluate

# Check GPU Availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[Med-Drishti ML] Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Define Dataset Class for Medical Prescriptions & Documents
class MedicalOCRDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_target_length=128):
        self.df = df
        self.img_dir = img_dir
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df.iloc[idx]['file_name']
        text = self.df.iloc[idx]['text']
        image_path = os.path.join(self.img_dir, file_name)
        
        # Open Image
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        
        # Encode Text
        labels = self.processor.tokenizer(text, padding="max_length", max_length=self.max_target_length).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding

In [ ]:
# Step 3: Load Model & Processor
MODEL_NAME = "microsoft/trocr-base-stage1"
print(f"Loading base model: {MODEL_NAME}")

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Set configuration tokens for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Beam Search Parameters for Precision
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 128
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

In [ ]:
# Step 4: Define Evaluation Metric (Character Error Rate & Word Error Rate)
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

In [ ]:
# Step 5: Configure Training Arguments for Kaggle GPU
OUTPUT_DIR = "/kaggle/working/med_drishti_trocr_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),  # Enable Mixed Precision for T4 GPUs
    predict_with_generate=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none"
)

print("Training Arguments Initialized:")
print(f"- Output Directory: {OUTPUT_DIR}")
print(f"- Batch Size: {training_args.per_device_train_batch_size}")
print(f"- FP16 Enabled: {training_args.fp16}")

In [ ]:
# Step 6: Save Model Artifacts for Med-Drishti Backend Integration
def export_model_for_backend(model, processor, export_path):
    os.makedirs(export_path, exist_ok=True)
    model.save_pretrained(export_path)
    processor.save_pretrained(export_path)
    print(f"✓ Fine-tuned TrOCR model successfully exported to: {export_path}")

# Call after training completion:
# export_model_for_backend(model, processor, "/kaggle/working/med_drishti_ocr_final")